# Compost Pricing Analysis: Market Survey and Statistical Comparison

This notebook provides a framework for:
1. Finding composting facilities using Google Custom Search API
2. Collecting pricing data with contextual information
3. Analyzing price variability
4. Comparing prices across categorical variables

**Requirements:**
- Google Custom Search API key
- Custom Search Engine ID

**Setup instructions:** https://developers.google.com/custom-search/v1/overview

In [ ]:
# Install required packages
!pip install google-api-python-client pandas numpy scipy matplotlib seaborn requests beautifulsoup4 -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from googleapiclient.discovery import build
import requests
from bs4 import BeautifulSoup
import re
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Configuration and API Setup

In [ ]:
# API Configuration
# Get your API key from: https://console.cloud.google.com/apis/credentials
# Get your CSE ID from: https://programmablesearchengine.google.com/

API_KEY = ''  # Your Google API key
CSE_ID = ''   # Your Custom Search Engine ID

# Search parameters
SEARCH_QUERIES = [
    'composting facility prices cubic yard',
    'compost sales price per yard',
    'bulk compost pricing',
    'commercial composting facility pricing'
]

# Geographic regions to focus on (optional)
TARGET_REGIONS = [
    'North Carolina',
    'California',
    'Washington',
    'New York',
    'Texas',
    'Florida'
]

## 2. Google Custom Search Functions

In [ ]:
def google_search(query, api_key, cse_id, num_results=10):
    """
    Perform Google Custom Search
    
    Args:
        query: Search query string
        api_key: Google API key
        cse_id: Custom Search Engine ID
        num_results: Number of results to return (max 10 per query)
    
    Returns:
        List of search results with titles, links, and snippets
    """
    service = build('customsearch', 'v1', developerKey=api_key)
    
    results = []
    start_index = 1
    
    while len(results) < num_results:
        try:
            res = service.cse().list(
                q=query,
                cx=cse_id,
                start=start_index
            ).execute()
            
            if 'items' in res:
                for item in res['items']:
                    results.append({
                        'title': item.get('title', ''),
                        'link': item.get('link', ''),
                        'snippet': item.get('snippet', ''),
                        'query': query
                    })
                    if len(results) >= num_results:
                        break
            else:
                break
                
            start_index += 10
            
        except Exception as e:
            print(f"Error searching for '{query}': {e}")
            break
    
    return results


def search_facilities_by_region(region, api_key, cse_id):
    """
    Search for composting facilities in a specific region
    """
    query = f"composting facility {region} pricing cubic yard"
    return google_search(query, api_key, cse_id, num_results=10)


def extract_prices_from_text(text):
    """
    Extract price information from text using regex
    Looks for patterns like: $XX, $XX.XX, XX per yard, etc.
    """
    # Pattern for prices
    price_patterns = [
        r'\$\s*(\d+(?:\.\d{2})?)\s*(?:per)?\s*(?:cubic)?\s*(?:yard|cy|cu\.?\s*yd)',
        r'(\d+(?:\.\d{2})?)\s*(?:dollars)?\s*/\s*(?:cubic)?\s*(?:yard|cy)',
        r'(?:cubic)?\s*(?:yard|cy)\s*[:-]?\s*\$?\s*(\d+(?:\.\d{2})?)',
    ]
    
    prices = []
    for pattern in price_patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE)
        for match in matches:
            try:
                price = float(match.group(1))
                if 5 <= price <= 200:  # Reasonable range filter
                    prices.append(price)
            except (ValueError, IndexError):
                continue
    
    return prices


def simple_page_scrape(url, timeout=10):
    """
    Simple scraping function to get page content
    Note: Respect robots.txt and rate limits in production
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Research Bot for Academic Study)'
        }
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()
        
        text = soup.get_text()
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = ' '.join(chunk for chunk in chunks if chunk)
        
        return text
    
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

## 3. Run Search and Collect URLs

In [ ]:
# Collect search results
all_results = []

if API_KEY and CSE_ID:
    print("Searching for composting facilities...\n")
    
    # General searches
    for query in SEARCH_QUERIES:
        print(f"Searching: {query}")
        results = google_search(query, API_KEY, CSE_ID, num_results=10)
        all_results.extend(results)
        print(f"Found {len(results)} results\n")
    
    # Regional searches
    for region in TARGET_REGIONS:
        print(f"Searching: {region}")
        results = search_facilities_by_region(region, API_KEY, CSE_ID)
        all_results.extend(results)
        print(f"Found {len(results)} results\n")
    
    # Create DataFrame
    search_df = pd.DataFrame(all_results)
    
    # Remove duplicates
    search_df = search_df.drop_duplicates(subset=['link'])
    
    print(f"\nTotal unique URLs found: {len(search_df)}")
    print("\nSample results:")
    print(search_df[['title', 'link']].head(10))
    
    # Save search results
    search_df.to_csv('compost_facility_urls.csv', index=False)
    print("\nSearch results saved to 'compost_facility_urls.csv'")
else:
    print("Please set API_KEY and CSE_ID to run searches")
    print("Loading from saved file if available...")
    try:
        search_df = pd.read_csv('compost_facility_urls.csv')
        print(f"Loaded {len(search_df)} URLs from file")
    except FileNotFoundError:
        print("No saved search results found")
        search_df = pd.DataFrame()

## 4. Data Collection Structure

### Define the data schema for manual/semi-automated collection

In [ ]:
# Data schema for compost pricing
pricing_schema = {
    'facility_name': 'string',
    'url': 'string',
    'location_city': 'string',
    'location_state': 'string',
    'region': 'string',  # Northeast, Southeast, Midwest, Southwest, West Coast
    'price_per_cy': 'float',
    'lbs_per_cy': 'float',  # Density in pounds per cubic yard
    'product_type': 'string',  # Commodity, Premium, Specialty Blend
    'quality_certification': 'string',  # STA Certified, Organic, None, etc.
    'feedstock_primary': 'string',  # Yard waste, Food waste, Manure, Mixed, etc.
    'processing_method': 'string',  # Windrow, ASP, In-vessel, Unknown
    'delivery_option': 'string',  # Pickup only, Delivery available, Both
    'minimum_order_cy': 'float',
    'volume_discount': 'boolean',
    'market_type': 'string',  # Urban, Suburban, Rural
    'facility_type': 'string',  # Municipal, Private, Farm-based
    'annual_production_tons': 'float',  # If available
    'maturity': 'string',  # Finished, Semi-finished, Fresh
    'moisture_content_pct': 'float',  # If available
    'organic_matter_pct': 'float',  # If available
    'cn_ratio': 'float',  # Carbon to Nitrogen ratio if available
    'notes': 'string',
    'data_collection_date': 'datetime',
    'price_date': 'datetime'  # When was this price effective
}

# Create empty DataFrame with schema
pricing_data = pd.DataFrame(columns=list(pricing_schema.keys()))

print("Data collection schema:")
print("\nRequired fields:")
print("- facility_name, location_state, price_per_cy, product_type")
print("\nOptional contextual fields:")
print("- lbs_per_cy (density), quality_certification, feedstock_primary")
print("- processing_method, market_type, facility_type, maturity")
print("- moisture_content_pct, organic_matter_pct, cn_ratio")

## 5. Automated Price Extraction (Experimental)

This attempts to extract prices from web pages automatically. Results will need manual verification.

In [ ]:
def extract_pricing_info(url):
    """
    Attempt to extract pricing information from a URL
    Returns a dictionary with extracted information
    """
    text = simple_page_scrape(url)
    
    if text is None:
        return None
    
    # Extract prices
    prices = extract_prices_from_text(text)
    
    # Look for density information
    density_pattern = r'(\d+)\s*(?:lbs?|pounds?)\s*(?:per)?\s*(?:cubic)?\s*(?:yard|cy)'
    density_matches = re.findall(density_pattern, text, re.IGNORECASE)
    densities = [float(d) for d in density_matches if 600 <= float(d) <= 1500]
    
    # Look for product types
    product_keywords = {
        'premium': ['premium', 'deluxe', 'professional', 'certified'],
        'commodity': ['bulk', 'standard', 'basic', 'economy'],
        'specialty': ['blend', 'specialty', 'custom', 'enriched', 'enhanced']
    }
    
    detected_products = []
    text_lower = text.lower()
    for product_type, keywords in product_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            detected_products.append(product_type)
    
    return {
        'url': url,
        'prices_found': prices,
        'densities_found': densities,
        'product_types_detected': detected_products,
        'text_sample': text[:500]  # First 500 chars for context
    }


# Run automated extraction on a sample
if not search_df.empty:
    sample_size = min(20, len(search_df))
    sample_urls = search_df['link'].head(sample_size).tolist()
    
    print(f"Attempting automated extraction on {sample_size} URLs...\n")
    
    extraction_results = []
    for i, url in enumerate(sample_urls, 1):
        print(f"Processing {i}/{sample_size}: {url[:60]}...")
        result = extract_pricing_info(url)
        if result:
            extraction_results.append(result)
    
    extraction_df = pd.DataFrame(extraction_results)
    
    print(f"\nExtracted data from {len(extraction_df)} pages")
    print(f"Found prices on {len([r for r in extraction_results if r and r['prices_found']])} pages")
    
    # Save extraction results
    extraction_df.to_csv('automated_extraction_results.csv', index=False)
    print("Results saved to 'automated_extraction_results.csv'")
else:
    print("No URLs available for extraction")

## 6. Manual Data Entry Template

### Load or create your pricing dataset here

In [ ]:
# Example data for demonstration
# Replace this with your actual collected data

sample_data = [
    {
        'facility_name': 'Green Cycle Composting',
        'location_state': 'NC',
        'region': 'Southeast',
        'price_per_cy': 35.00,
        'lbs_per_cy': 900,
        'product_type': 'Commodity',
        'quality_certification': 'None',
        'feedstock_primary': 'Yard waste',
        'processing_method': 'Windrow',
        'market_type': 'Suburban',
        'facility_type': 'Municipal',
        'maturity': 'Finished'
    },
    {
        'facility_name': 'Premium Organics',
        'location_state': 'CA',
        'region': 'West Coast',
        'price_per_cy': 65.00,
        'lbs_per_cy': 1000,
        'product_type': 'Premium',
        'quality_certification': 'STA Certified',
        'feedstock_primary': 'Mixed',
        'processing_method': 'ASP',
        'market_type': 'Urban',
        'facility_type': 'Private',
        'maturity': 'Finished'
    },
    # Add more facilities as you collect data
]

# Option 1: Start with sample data
pricing_data = pd.DataFrame(sample_data)

# Option 2: Load from CSV if you have existing data
# pricing_data = pd.read_csv('compost_pricing_data.csv')

# Option 3: Load from Google Sheets (requires gspread)
# import gspread
# from google.colab import auth
# auth.authenticate_user()
# gc = gspread.authorize(GoogleCredentials.get_application_default())
# sheet = gc.open('Compost Pricing Data').sheet1
# pricing_data = pd.DataFrame(sheet.get_all_records())

print(f"Loaded {len(pricing_data)} facility records")
print("\nData preview:")
print(pricing_data.head())

## 7. Data Cleaning and Preparation

In [ ]:
# Clean and prepare data
def clean_pricing_data(df):
    """
    Clean and standardize pricing data
    """
    df = df.copy()
    
    # Convert numeric columns
    numeric_cols = ['price_per_cy', 'lbs_per_cy', 'minimum_order_cy', 
                    'annual_production_tons', 'moisture_content_pct', 
                    'organic_matter_pct', 'cn_ratio']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Standardize categorical variables
    if 'product_type' in df.columns:
        df['product_type'] = df['product_type'].str.title()
    
    if 'region' in df.columns:
        df['region'] = df['region'].str.title()
    
    if 'market_type' in df.columns:
        df['market_type'] = df['market_type'].str.title()
    
    # Calculate price per pound if density is available
    if 'lbs_per_cy' in df.columns and 'price_per_cy' in df.columns:
        df['price_per_lb'] = df['price_per_cy'] / df['lbs_per_cy']
    
    # Remove outliers (prices outside reasonable range)
    if 'price_per_cy' in df.columns:
        df = df[(df['price_per_cy'] >= 10) & (df['price_per_cy'] <= 150)]
    
    return df


pricing_data_clean = clean_pricing_data(pricing_data)

print("Data cleaned and prepared")
print(f"\nRecords after cleaning: {len(pricing_data_clean)}")
print("\nData types:")
print(pricing_data_clean.dtypes)
print("\nMissing values:")
print(pricing_data_clean.isnull().sum())

## 8. Descriptive Statistics and Variability Analysis

In [ ]:
# Overall price statistics
def calculate_price_statistics(df, price_col='price_per_cy'):
    """
    Calculate comprehensive price statistics
    """
    stats_dict = {
        'Mean': df[price_col].mean(),
        'Median': df[price_col].median(),
        'Std Dev': df[price_col].std(),
        'CV (%)': (df[price_col].std() / df[price_col].mean()) * 100,
        'Min': df[price_col].min(),
        'Max': df[price_col].max(),
        'Range': df[price_col].max() - df[price_col].min(),
        'Q1': df[price_col].quantile(0.25),
        'Q3': df[price_col].quantile(0.75),
        'IQR': df[price_col].quantile(0.75) - df[price_col].quantile(0.25),
        'Skewness': df[price_col].skew(),
        'Kurtosis': df[price_col].kurtosis()
    }
    
    return pd.Series(stats_dict)


print("=" * 60)
print("OVERALL PRICE STATISTICS ($/cubic yard)")
print("=" * 60)
overall_stats = calculate_price_statistics(pricing_data_clean)
print(overall_stats.to_string())
print("\n" + "=" * 60)

# Visualize price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(pricing_data_clean['price_per_cy'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(overall_stats['Mean'], color='red', linestyle='--', linewidth=2, label=f"Mean: ${overall_stats['Mean']:.2f}")
axes[0].axvline(overall_stats['Median'], color='green', linestyle='--', linewidth=2, label=f"Median: ${overall_stats['Median']:.2f}")
axes[0].set_xlabel('Price per Cubic Yard ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Compost Prices', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot
axes[1].boxplot(pricing_data_clean['price_per_cy'], vert=True)
axes[1].set_ylabel('Price per Cubic Yard ($)', fontsize=12)
axes[1].set_title('Price Variability (Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCoefficient of Variation: {overall_stats['CV (%)']:.2f}%")
if overall_stats['CV (%)'] < 20:
    print("Low variability - prices are relatively consistent")
elif overall_stats['CV (%)'] < 40:
    print("Moderate variability - prices show some variation")
else:
    print("High variability - prices vary significantly across facilities")

## 9. Price per Pound Analysis (Density Normalization)

In [ ]:
# Analyze price per pound if density data available
if 'price_per_lb' in pricing_data_clean.columns:
    density_data = pricing_data_clean.dropna(subset=['lbs_per_cy', 'price_per_lb'])
    
    if len(density_data) > 0:
        print("=" * 60)
        print("DENSITY AND PRICE PER POUND ANALYSIS")
        print("=" * 60)
        print(f"\nFacilities with density data: {len(density_data)}")
        print(f"\nDensity statistics (lbs/cy):")
        print(density_data['lbs_per_cy'].describe())
        print(f"\nPrice per pound statistics:")
        print(density_data['price_per_lb'].describe())
        
        # Visualize relationship
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Scatter plot: density vs price per cy
        axes[0].scatter(density_data['lbs_per_cy'], density_data['price_per_cy'], alpha=0.6, s=100)
        axes[0].set_xlabel('Density (lbs/cubic yard)', fontsize=12)
        axes[0].set_ylabel('Price per Cubic Yard ($)', fontsize=12)
        axes[0].set_title('Price vs. Density', fontsize=14, fontweight='bold')
        axes[0].grid(alpha=0.3)
        
        # Calculate correlation
        corr = density_data['lbs_per_cy'].corr(density_data['price_per_cy'])
        axes[0].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                     transform=axes[0].transAxes, fontsize=11,
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Price per pound distribution
        axes[1].hist(density_data['price_per_lb'], bins=15, edgecolor='black', alpha=0.7, color='green')
        axes[1].axvline(density_data['price_per_lb'].mean(), color='red', linestyle='--', 
                       linewidth=2, label=f"Mean: ${density_data['price_per_lb'].mean():.4f}/lb")
        axes[1].set_xlabel('Price per Pound ($)', fontsize=12)
        axes[1].set_ylabel('Frequency', fontsize=12)
        axes[1].set_title('Distribution of Price per Pound', fontsize=14, fontweight='bold')
        axes[1].legend()
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('density_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"\nPrice per pound provides density-normalized comparison")
        print(f"Mean: ${density_data['price_per_lb'].mean():.4f}/lb")
        print(f"Range: ${density_data['price_per_lb'].min():.4f} - ${density_data['price_per_lb'].max():.4f}/lb")
    else:
        print("\nNo density data available for analysis")
else:
    print("\nNo density data in dataset - collect lbs_per_cy to enable this analysis")

## 10. Categorical Comparisons

### Compare prices across different categories with statistical tests

In [ ]:
def compare_categories(df, category_col, price_col='price_per_cy', min_group_size=2):
    """
    Compare prices across categorical variable with visualizations and statistics
    """
    # Filter out categories with insufficient data
    category_counts = df[category_col].value_counts()
    valid_categories = category_counts[category_counts >= min_group_size].index
    df_filtered = df[df[category_col].isin(valid_categories)].copy()
    
    if len(df_filtered) < min_group_size or df_filtered[category_col].nunique() < 2:
        print(f"Insufficient data for {category_col} comparison")
        return None
    
    print("=" * 70)
    print(f"PRICE COMPARISON BY {category_col.upper().replace('_', ' ')}")
    print("=" * 70)
    
    # Descriptive statistics by category
    grouped_stats = df_filtered.groupby(category_col)[price_col].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std Dev', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(2)
    
    print("\nDescriptive Statistics:")
    print(grouped_stats.to_string())
    
    # Statistical tests
    groups = [group[price_col].values for name, group in df_filtered.groupby(category_col)]
    
    if len(groups) == 2:
        # T-test for two groups
        t_stat, p_value = stats.ttest_ind(*groups)
        test_name = "Independent t-test"
    else:
        # ANOVA for multiple groups
        t_stat, p_value = stats.f_oneway(*groups)
        test_name = "One-way ANOVA"
    
    print(f"\n{test_name}:")
    print(f"  Test statistic: {t_stat:.4f}")
    print(f"  P-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"  ✓ Significant difference detected (p < 0.05)")
        print(f"    Prices vary significantly across {category_col}")
    else:
        print(f"  ✗ No significant difference (p >= 0.05)")
        print(f"    Prices are similar across {category_col}")
    
    # Effect size (Cohen's d for two groups, eta-squared for ANOVA)
    if len(groups) == 2:
        cohens_d = (np.mean(groups[0]) - np.mean(groups[1])) / np.sqrt((np.std(groups[0])**2 + np.std(groups[1])**2) / 2)
        print(f"  Cohen's d: {cohens_d:.3f}")
        if abs(cohens_d) < 0.2:
            effect = "negligible"
        elif abs(cohens_d) < 0.5:
            effect = "small"
        elif abs(cohens_d) < 0.8:
            effect = "medium"
        else:
            effect = "large"
        print(f"  Effect size: {effect}")
    
    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot
    df_filtered.boxplot(column=price_col, by=category_col, ax=axes[0])
    axes[0].set_xlabel(category_col.replace('_', ' ').title(), fontsize=12)
    axes[0].set_ylabel('Price per Cubic Yard ($)', fontsize=12)
    axes[0].set_title(f'Price Distribution by {category_col.replace("_", " ").title()}', fontsize=14, fontweight='bold')
    plt.sca(axes[0])
    plt.xticks(rotation=45, ha='right')
    
    # Bar plot with error bars
    grouped_stats[['Mean']].plot(kind='bar', ax=axes[1], yerr=grouped_stats['Std Dev'], 
                                  capsize=4, alpha=0.7, color='steelblue')
    axes[1].set_xlabel(category_col.replace('_', ' ').title(), fontsize=12)
    axes[1].set_ylabel('Mean Price per Cubic Yard ($)', fontsize=12)
    axes[1].set_title('Mean Prices with Standard Deviation', fontsize=14, fontweight='bold')
    axes[1].legend().set_visible(False)
    plt.sca(axes[1])
    plt.xticks(rotation=45, ha='right')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    filename = f'comparison_{category_col}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    
    return grouped_stats


# Run comparisons for all relevant categorical variables
categorical_vars = ['product_type', 'region', 'market_type', 'facility_type', 
                   'processing_method', 'quality_certification', 'feedstock_primary', 'maturity']

comparison_results = {}

for cat_var in categorical_vars:
    if cat_var in pricing_data_clean.columns:
        result = compare_categories(pricing_data_clean, cat_var)
        if result is not None:
            comparison_results[cat_var] = result
            print("\n")

## 11. Multi-factor Analysis

In [ ]:
# Analyze interaction between multiple factors
def multifactor_analysis(df, factor1, factor2, price_col='price_per_cy'):
    """
    Analyze price variation across two categorical factors
    """
    if factor1 not in df.columns or factor2 not in df.columns:
        print(f"Missing factors: {factor1} or {factor2}")
        return
    
    # Create pivot table
    pivot = df.pivot_table(values=price_col, 
                          index=factor1, 
                          columns=factor2, 
                          aggfunc=['mean', 'count'])
    
    print("=" * 70)
    print(f"MULTI-FACTOR ANALYSIS: {factor1.upper()} × {factor2.upper()}")
    print("=" * 70)
    print("\nMean Prices:")
    print(pivot['mean'].round(2).to_string())
    print("\nSample Counts:")
    print(pivot['count'].to_string())
    
    # Visualize heatmap if sufficient data
    if pivot['mean'].notna().sum().sum() >= 4:  # At least 4 valid cells
        plt.figure(figsize=(10, 6))
        sns.heatmap(pivot['mean'], annot=True, fmt='.1f', cmap='RdYlGn_r', 
                   cbar_kws={'label': 'Price ($/cy)'}, linewidths=0.5)
        plt.title(f'Mean Compost Prices by {factor1.replace("_", " ").title()} and {factor2.replace("_", " ").title()}',
                 fontsize=14, fontweight='bold', pad=20)
        plt.xlabel(factor2.replace('_', ' ').title(), fontsize=12)
        plt.ylabel(factor1.replace('_', ' ').title(), fontsize=12)
        plt.tight_layout()
        plt.savefig(f'multifactor_{factor1}_{factor2}.png', dpi=300, bbox_inches='tight')
        plt.show()


# Example multi-factor analyses
if 'product_type' in pricing_data_clean.columns and 'region' in pricing_data_clean.columns:
    multifactor_analysis(pricing_data_clean, 'product_type', 'region')
    print("\n")

if 'market_type' in pricing_data_clean.columns and 'facility_type' in pricing_data_clean.columns:
    multifactor_analysis(pricing_data_clean, 'market_type', 'facility_type')
    print("\n")

## 12. Summary Report

In [ ]:
# Generate comprehensive summary report
def generate_summary_report(df, comparison_results):
    """
    Generate a comprehensive summary report
    """
    report = []
    report.append("=" * 80)
    report.append("COMPOST PRICING ANALYSIS - SUMMARY REPORT")
    report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("=" * 80)
    report.append("")
    
    # Dataset overview
    report.append("DATASET OVERVIEW")
    report.append("-" * 40)
    report.append(f"Total facilities: {len(df)}")
    report.append(f"States represented: {df['location_state'].nunique() if 'location_state' in df.columns else 'N/A'}")
    report.append(f"Regions represented: {df['region'].nunique() if 'region' in df.columns else 'N/A'}")
    report.append("")
    
    # Price overview
    report.append("PRICE OVERVIEW ($/cubic yard)")
    report.append("-" * 40)
    report.append(f"Mean: ${df['price_per_cy'].mean():.2f}")
    report.append(f"Median: ${df['price_per_cy'].median():.2f}")
    report.append(f"Std Dev: ${df['price_per_cy'].std():.2f}")
    report.append(f"Range: ${df['price_per_cy'].min():.2f} - ${df['price_per_cy'].max():.2f}")
    report.append(f"Coefficient of Variation: {(df['price_per_cy'].std() / df['price_per_cy'].mean() * 100):.2f}%")
    report.append("")
    
    # Key findings from categorical comparisons
    report.append("KEY FINDINGS FROM CATEGORICAL COMPARISONS")
    report.append("-" * 40)
    
    for category, stats in comparison_results.items():
        report.append(f"\n{category.replace('_', ' ').title()}:")
        highest = stats['Mean'].idxmax()
        lowest = stats['Mean'].idxmin()
        report.append(f"  Highest: {highest} (${stats.loc[highest, 'Mean']:.2f})")
        report.append(f"  Lowest: {lowest} (${stats.loc[lowest, 'Mean']:.2f})")
        report.append(f"  Difference: ${stats.loc[highest, 'Mean'] - stats.loc[lowest, 'Mean']:.2f}")
    
    report.append("")
    report.append("=" * 80)
    
    report_text = "\n".join(report)
    print(report_text)
    
    # Save report
    with open('compost_pricing_summary_report.txt', 'w') as f:
        f.write(report_text)
    print("\nReport saved to 'compost_pricing_summary_report.txt'")
    
    return report_text


summary_report = generate_summary_report(pricing_data_clean, comparison_results)

## 13. Export Results

In [ ]:
# Export all results
print("Exporting results...")

# Save cleaned dataset
pricing_data_clean.to_csv('compost_pricing_clean.csv', index=False)
print("✓ Clean dataset: compost_pricing_clean.csv")

# Save comparison results
with pd.ExcelWriter('compost_pricing_analysis.xlsx') as writer:
    pricing_data_clean.to_excel(writer, sheet_name='Data', index=False)
    
    for category, stats in comparison_results.items():
        sheet_name = category[:31]  # Excel sheet name limit
        stats.to_excel(writer, sheet_name=sheet_name)

print("✓ Analysis workbook: compost_pricing_analysis.xlsx")
print("✓ Summary report: compost_pricing_summary_report.txt")
print("✓ Visualizations: *.png files")
print("\nAll results exported successfully!")

## 14. Next Steps and Recommendations

### Recommendations for Expanding This Analysis:

1. **Increase Sample Size**
   - Target: 50+ facilities across multiple regions
   - Focus on underrepresented regions/categories
   - Include seasonal pricing variations

2. **Collect Additional Context**
   - Nutrient content (N-P-K analysis)
   - Contaminant levels (plastic, metal)
   - Maturity indicators (respiration rate, germination index)
   - Local market conditions (competitors, demand)

3. **Advanced Modeling**
   - Build predictive pricing model using regression
   - Hedonic pricing model for product differentiation
   - Geographic pricing clusters using spatial analysis
   - Time series analysis if you collect temporal data

4. **Integration with Your Consulting Business**
   - Use as a benchmarking tool for clients
   - Develop region-specific pricing guidance
   - Create pricing optimization recommendations
   - Build market entry analysis for new facilities

5. **Data Quality Improvements**
   - Standardize density measurements (lbs/cy)
   - Verify certifications directly with facilities
   - Track seasonal pricing changes
   - Note delivery vs. pickup pricing differences